In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Simple Survival Analysis with Monte Carlo - C-INDEX ONLY
=========================================================

Chỉ hiển thị kết quả C-index cho mô hình Dynamic Features Monte Carlo
Không tạo bất kỳ phân tích khác

Author: Claude
Date: 2025-11-17
"""

import os
import json
import numpy as np
import pandas as pd
from typing import Dict
import warnings

from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

print("""
╔═══════════════════════════════════════════════════════════════════════════════╗
║      SURVIVAL ANALYSIS - MONTE CARLO (C-INDEX ONLY)                          ║
║                   Patient ID Matching + Years (Real Time)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

# ============================================================================
# CONFIGURATION
# ============================================================================

RANDOM_SEED = 42
N_MC_RUNS = 500  # Monte Carlo runs per patient
T_MAX = 15       # Time horizon in YEARS

OUTPUT_DIR = 'data3d/results/simple_survival_mc_cindex_only'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nCONFIGURATION:")
print(f"  • Monte Carlo runs: {N_MC_RUNS}")
print(f"  • Time horizon: {T_MAX} years")
print(f"  • Random seed: {RANDOM_SEED}")


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def days_to_years(days: float) -> float:
    """Convert days to years."""
    if pd.isna(days):
        return np.nan
    return days / 365.25


def sigmoid(t: np.ndarray, L: float, k: float, t0: float) -> np.ndarray:
    """Sigmoid function for response modeling."""
    return L / (1 + np.exp(-k * (t - t0)))


def monte_carlo_from_radiomics(radiomics_vector: np.ndarray,
                               n_runs: int = 500,
                               T: int = 15,
                               seed: int = None) -> Dict:
    """
    Transform static radiomics into dynamic features via Monte Carlo.
    Returns: auc, slope, time_half, peak, uncertainty
    """
    if seed is not None:
        np.random.seed(seed)

    x = radiomics_vector
    t = np.linspace(0, T, T)
    curves = []

    for _ in range(n_runs):
        # Random projection weights
        w_L = np.random.randn(len(x))
        w_k = np.random.randn(len(x))
        w_t0 = np.random.randn(len(x))

        # Generate sigmoid parameters
        L = np.dot(w_L, x)
        L = abs(L) + 1e-3

        k_raw = np.dot(w_k, x)
        k = 0.1 + 0.9 * (1 / (1 + np.exp(-k_raw)))

        t0_raw = np.dot(w_t0, x)
        t0 = 0.5 + 13.5 * (1 / (1 + np.exp(-t0_raw)))

        # Generate sigmoid curve
        S = sigmoid(t, L, k, t0)
        curves.append(S)

    curves = np.array(curves)

    # Aggregate statistics
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)

    # Extract dynamic features
    gradient = np.gradient(mean_curve)
    slope = np.max(gradient)
    auc = np.trapz(mean_curve, t)

    half_max = 0.5 * mean_curve.max()
    time_half_idx = np.argmax(mean_curve >= half_max)
    time_half = t[time_half_idx] if time_half_idx > 0 else 0

    peak_idx = np.argmax(gradient)
    peak = mean_curve[peak_idx]

    uncertainty = std_curve.mean()

    return {
        "auc": auc,
        "slope": slope,
        "time_half": time_half,
        "peak": peak,
        "uncertainty": uncertainty
    }


def generate_dynamic_features(radiomics_features: np.ndarray,
                             n_runs: int = 500,
                             T: int = 15) -> pd.DataFrame:
    """Generate dynamic features for all patients via Monte Carlo."""
    n_patients = radiomics_features.shape[0]
    dynamic_features = []

    print(f"\n⚙️  Generating dynamic features for {n_patients} patients...")

    for i in range(n_patients):
        if (i + 1) % 100 == 0:
            print(f"  Processing: {i+1}/{n_patients}")

        radiomics_vector = radiomics_features[i]
        result = monte_carlo_from_radiomics(
            radiomics_vector,
            n_runs=n_runs,
            T=T,
            seed=RANDOM_SEED + i
        )

        dynamic_features.append([
            result["auc"],
            result["slope"],
            result["time_half"],
            result["peak"],
            result["uncertainty"]
        ])

    df = pd.DataFrame(
        dynamic_features,
        columns=["dynamic_auc", "dynamic_slope", "dynamic_time_half",
                 "dynamic_peak", "dynamic_uncertainty"]
    )

    print(f"  ✓ Generated: {df.shape}")
    return df


# ============================================================================
# STEP 1: LOAD DATA WITH PATIENT ID MATCHING
# ============================================================================

print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# Load radiomics features
features_path = 'visualization_output/features_normalized_standard.npy'
if not os.path.exists(features_path):
    features_path = 'visualization_output/radiomic_features_440.npy'

print(f"\n📂 Radiomics: {features_path}")
radiomics_features_all = np.load(features_path)
print(f"   Shape: {radiomics_features_all.shape}")

# Load feature names
names_path = 'visualization_output/feature_names_normalized_standard.json'
with open(names_path, 'r') as f:
    feature_names = json.load(f)

# Load patient IDs
ids_path = 'visualization_output/patient_ids.json'
with open(ids_path, 'r') as f:
    patient_ids_all = json.load(f)
print(f"   Patient IDs: {len(patient_ids_all)}")

# Load clinical data
clinical_paths = [
    "nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv",
    "NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv",
]

clinical_data = None
for path in clinical_paths:
    if os.path.exists(path):
        clinical_data = pd.read_csv(path)
        print(f"📂 Clinical: {path}")
        break

if clinical_data is None:
    raise FileNotFoundError("Clinical data not found!")

# ============================================================================
# PATIENT ID MATCHING
# ============================================================================

print("\n" + "="*80)
print("STEP 2: PATIENT ID MATCHING")
print("="*80)

# Find patient ID column
patient_id_col = None
for col in clinical_data.columns:
    if 'patient' in col.lower() and 'id' in col.lower():
        patient_id_col = col
        break

if patient_id_col is None:
    clinical_data['PatientID'] = clinical_data.index.astype(str)
    patient_id_col = 'PatientID'

clinical_data['PatientID_str'] = clinical_data[patient_id_col].astype(str)
patient_ids_radiomics = [str(pid) for pid in patient_ids_all]

# Find survival columns
time_col = [c for c in clinical_data.columns if 'survival.time' in c.lower()][0]
event_col = [c for c in clinical_data.columns if 'event' in c.lower() or 'deadstatus' in c.lower()][0]

# Match patients
print("\n🔍 Matching patients...")
matched_data = []

for i, pid in enumerate(patient_ids_radiomics):
    clinical_row = clinical_data[clinical_data['PatientID_str'] == pid]

    if len(clinical_row) == 0:
        if i < len(clinical_data):
            clinical_row = clinical_data.iloc[[i]]
        else:
            continue

    if len(clinical_row) == 0:
        continue

    time_days = clinical_row[time_col].values[0]
    event = clinical_row[event_col].values[0]

    if pd.isna(time_days) or pd.isna(event):
        continue

    time_years = days_to_years(time_days)
    radiomics_vector = radiomics_features_all[i]

    matched_data.append({
        'patient_id': pid,
        'time_years': time_years,
        'event': event,
        'radiomics_vector': radiomics_vector
    })

print(f"✓ Matched {len(matched_data)} patients")

# Create aligned datasets
radiomics_features = np.array([d['radiomics_vector'] for d in matched_data])

survival_df = pd.DataFrame({
    'time': [d['time_years'] for d in matched_data],
    'event': [d['event'] for d in matched_data]
})

print(f"\n✅ DATA SUMMARY:")
print(f"   Patients: {len(matched_data)}")
print(f"   Events: {int(survival_df['event'].sum())}")
print(f"   Event rate: {survival_df['event'].mean()*100:.1f}%")
print(f"   Median survival: {survival_df['time'].median():.2f} years")


# ============================================================================
# STEP 2: GENERATE DYNAMIC FEATURES VIA MONTE CARLO
# ============================================================================

print("\n" + "="*80)
print("STEP 3: MONTE CARLO - GENERATE DYNAMIC FEATURES")
print("="*80)

# Standardize radiomics
scaler_mc = StandardScaler()
radiomics_standardized = scaler_mc.fit_transform(radiomics_features)

# Generate dynamic features
dynamic_df = generate_dynamic_features(
    radiomics_standardized,
    n_runs=N_MC_RUNS,
    T=T_MAX
)

# Standardize dynamic features
scaler_dynamic = StandardScaler()
dynamic_scaled = scaler_dynamic.fit_transform(dynamic_df)
dynamic_scaled_df = pd.DataFrame(dynamic_scaled, columns=dynamic_df.columns)

# ============================================================================
# STEP 3: TRAIN COX MODEL WITH DYNAMIC FEATURES ONLY
# ============================================================================

print("\n" + "="*80)
print("STEP 4: TRAINING COX MODEL - DYNAMIC FEATURES")
print("="*80)

# Prepare Cox data
cox_df = pd.concat([
    dynamic_scaled_df.reset_index(drop=True),
    survival_df[['time', 'event']].reset_index(drop=True)
], axis=1)

# Remove any NaN rows
cox_df = cox_df.dropna()

print(f"\nFeatures: {cox_df.shape[1] - 2} (5 dynamic features)")
print(f"Samples: {cox_df.shape[0]}")

# Train Cox model
cph = CoxPHFitter(penalizer=0.1)

try:
    cph.fit(cox_df, duration_col='time', event_col='event', show_progress=False)
    c_index = cph.concordance_index_
    
    print(f"\n✓ Model trained successfully!")
    
except Exception as e:
    print(f"✗ Error training model: {e}")
    c_index = None


# ============================================================================
# FINAL RESULTS - C-INDEX ONLY
# ============================================================================

print("\n" + "="*80)
print("🎯 RESULTS - C-INDEX ONLY")
print("="*80)

if c_index is not None:
    print(f"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                    MONTE CARLO DYNAMIC FEATURES MODEL                        ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  Model: Cox Proportional Hazards with 5 Dynamic Features                     ║
║                                                                               ║
║  Features (from Monte Carlo):                                                ║
║    • dynamic_auc                                                             ║
║    • dynamic_slope                                                           ║
║    • dynamic_time_half                                                       ║
║    • dynamic_peak                                                            ║
║    • dynamic_uncertainty                                                     ║
║                                                                               ║
║  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ║
║                                                                               ║
║  🏆 C-INDEX (CONCORDANCE INDEX): {c_index:.4f}                                           ║
║                                                                               ║
║  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ║
║                                                                               ║
║  Dataset:                                                                     ║
║    • Patients: {len(cox_df)}                                                               ║
║    • Events: {int(cox_df['event'].sum())}                                                               ║
║    • Median survival: {cox_df['time'].median():.2f} years                                               ║
║                                                                               ║
║  Monte Carlo:                                                                ║
║    • Runs per patient: {N_MC_RUNS}                                                    ║
║    • Time horizon: {T_MAX} years                                                     ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

    # Save simple result
    result_dict = {
        "model": "Cox Proportional Hazards (Dynamic Features)",
        "c_index": float(c_index),
        "n_patients": int(len(cox_df)),
        "n_events": int(cox_df['event'].sum()),
        "n_features": 5,
        "monte_carlo_runs": N_MC_RUNS,
        "time_horizon_years": T_MAX
    }
    
    result_path = os.path.join(OUTPUT_DIR, 'c_index_result.json')
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=2)
    
    print(f"\n💾 Results saved: {result_path}")

else:
    print("✗ Model training failed!")

print("\n" + "="*80 + "\n")


╔═══════════════════════════════════════════════════════════════════════════════╗
║      SURVIVAL ANALYSIS - MONTE CARLO (C-INDEX ONLY)                          ║
║                   Patient ID Matching + Years (Real Time)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝


CONFIGURATION:
  • Monte Carlo runs: 500
  • Time horizon: 15 years
  • Random seed: 42

STEP 1: LOADING DATA

📂 Radiomics: visualization_output/features_normalized_standard.npy
   Shape: (421, 261)
   Patient IDs: 421
📂 Clinical: nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv

STEP 2: PATIENT ID MATCHING

🔍 Matching patients...
✓ Matched 421 patients

✅ DATA SUMMARY:
   Patients: 421
   Events: 373
   Event rate: 88.6%
   Median survival: 1.50 years

STEP 3: MONTE CARLO - GENERATE DYNAMIC FEATURES

⚙️  Generating dynamic features for 421 patients...
  Processing: 100/421
  Processing: 200/421
  Processing: 300/421
  Processing: 400/421
  ✓ Generated

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Simple Survival Analysis with Monte Carlo - C-INDEX ONLY
=========================================================

Chỉ hiển thị kết quả C-index cho mô hình Dynamic Features Monte Carlo
Không tạo bất kỳ phân tích khác

Author: Claude
Date: 2025-11-17
"""

import os
import json
import numpy as np
import pandas as pd
from typing import Dict
import warnings

from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

print("""
╔═══════════════════════════════════════════════════════════════════════════════╗
║      SURVIVAL ANALYSIS - MONTE CARLO (C-INDEX ONLY)                          ║
║                   Patient ID Matching + Years (Real Time)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

# ============================================================================
# CONFIGURATION
# ============================================================================

RANDOM_SEED = 42
N_MC_RUNS = 500  # Monte Carlo runs per patient
T_MAX = 15       # Time horizon in YEARS

OUTPUT_DIR = 'results/simple_survival_mc_cindex_only'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nCONFIGURATION:")
print(f"  • Monte Carlo runs: {N_MC_RUNS}")
print(f"  • Time horizon: {T_MAX} years")
print(f"  • Random seed: {RANDOM_SEED}")


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def days_to_years(days: float) -> float:
    """Convert days to years."""
    if pd.isna(days):
        return np.nan
    return days / 365.25


def sigmoid(t: np.ndarray, L: float, k: float, t0: float) -> np.ndarray:
    """Sigmoid function for response modeling."""
    return L / (1 + np.exp(-k * (t - t0)))


def monte_carlo_from_radiomics(radiomics_vector: np.ndarray,
                               n_runs: int = 500,
                               T: int = 15,
                               seed: int = None) -> Dict:
    """
    Transform static radiomics into dynamic features via Monte Carlo.
    Returns: auc, slope, time_half, peak, uncertainty
    """
    if seed is not None:
        np.random.seed(seed)

    x = radiomics_vector
    t = np.linspace(0, T, T)
    curves = []

    for _ in range(n_runs):
        # Random projection weights
        w_L = np.random.randn(len(x))
        w_k = np.random.randn(len(x))
        w_t0 = np.random.randn(len(x))

        # Generate sigmoid parameters
        L = np.dot(w_L, x)
        L = abs(L) + 1e-3

        k_raw = np.dot(w_k, x)
        k = 0.1 + 0.9 * (1 / (1 + np.exp(-k_raw)))

        t0_raw = np.dot(w_t0, x)
        t0 = 0.5 + 13.5 * (1 / (1 + np.exp(-t0_raw)))

        # Generate sigmoid curve
        S = sigmoid(t, L, k, t0)
        curves.append(S)

    curves = np.array(curves)

    # Aggregate statistics
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)

    # Extract dynamic features
    gradient = np.gradient(mean_curve)
    slope = np.max(gradient)
    auc = np.trapz(mean_curve, t)

    half_max = 0.5 * mean_curve.max()
    time_half_idx = np.argmax(mean_curve >= half_max)
    time_half = t[time_half_idx] if time_half_idx > 0 else 0

    peak_idx = np.argmax(gradient)
    peak = mean_curve[peak_idx]

    uncertainty = std_curve.mean()

    return {
        "auc": auc,
        "slope": slope,
        "time_half": time_half,
        "peak": peak,
        "uncertainty": uncertainty
    }


def generate_dynamic_features(radiomics_features: np.ndarray,
                             n_runs: int = 500,
                             T: int = 15) -> pd.DataFrame:
    """Generate dynamic features for all patients via Monte Carlo."""
    n_patients = radiomics_features.shape[0]
    dynamic_features = []

    print(f"\n⚙️  Generating dynamic features for {n_patients} patients...")

    for i in range(n_patients):
        if (i + 1) % 100 == 0:
            print(f"  Processing: {i+1}/{n_patients}")

        radiomics_vector = radiomics_features[i]
        result = monte_carlo_from_radiomics(
            radiomics_vector,
            n_runs=n_runs,
            T=T,
            seed=RANDOM_SEED + i
        )

        dynamic_features.append([
            result["auc"],
            result["slope"],
            result["time_half"],
            result["peak"],
            result["uncertainty"]
        ])

    df = pd.DataFrame(
        dynamic_features,
        columns=["dynamic_auc", "dynamic_slope", "dynamic_time_half",
                 "dynamic_peak", "dynamic_uncertainty"]
    )

    print(f"  ✓ Generated: {df.shape}")
    return df


# ============================================================================
# STEP 1: LOAD DATA WITH PATIENT ID MATCHING
# ============================================================================

print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# Load radiomics features
features_path = 'visualization_output/features_normalized_standard.npy'
if not os.path.exists(features_path):
    features_path = 'visualization_output/radiomic_features_440.npy'

print(f"\n📂 Radiomics: {features_path}")
radiomics_features_all = np.load(features_path)
print(f"   Shape: {radiomics_features_all.shape}")

# Load feature names
names_path = 'visualization_output/feature_names_normalized_standard.json'
with open(names_path, 'r') as f:
    feature_names = json.load(f)

# Load patient IDs
ids_path = 'visualization_output/patient_ids.json'
with open(ids_path, 'r') as f:
    patient_ids_all = json.load(f)
print(f"   Patient IDs: {len(patient_ids_all)}")

# Load clinical data
clinical_paths = [
    "nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv",
    "NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv",
]

clinical_data = None
for path in clinical_paths:
    if os.path.exists(path):
        clinical_data = pd.read_csv(path)
        print(f"📂 Clinical: {path}")
        break

if clinical_data is None:
    raise FileNotFoundError("Clinical data not found!")

# ============================================================================
# PATIENT ID MATCHING
# ============================================================================

print("\n" + "="*80)
print("STEP 2: PATIENT ID MATCHING")
print("="*80)

# Find patient ID column
patient_id_col = None
for col in clinical_data.columns:
    if 'patient' in col.lower() and 'id' in col.lower():
        patient_id_col = col
        break

if patient_id_col is None:
    clinical_data['PatientID'] = clinical_data.index.astype(str)
    patient_id_col = 'PatientID'

clinical_data['PatientID_str'] = clinical_data[patient_id_col].astype(str)
patient_ids_radiomics = [str(pid) for pid in patient_ids_all]

# Find survival columns
time_col = [c for c in clinical_data.columns if 'survival.time' in c.lower()][0]
event_col = [c for c in clinical_data.columns if 'event' in c.lower() or 'deadstatus' in c.lower()][0]

# Match patients
print("\n🔍 Matching patients...")
matched_data = []

for i, pid in enumerate(patient_ids_radiomics):
    clinical_row = clinical_data[clinical_data['PatientID_str'] == pid]

    if len(clinical_row) == 0:
        if i < len(clinical_data):
            clinical_row = clinical_data.iloc[[i]]
        else:
            continue

    if len(clinical_row) == 0:
        continue

    time_days = clinical_row[time_col].values[0]
    event = clinical_row[event_col].values[0]

    if pd.isna(time_days) or pd.isna(event):
        continue

    time_years = days_to_years(time_days)
    radiomics_vector = radiomics_features_all[i]

    matched_data.append({
        'patient_id': pid,
        'time_years': time_years,
        'event': event,
        'radiomics_vector': radiomics_vector
    })

print(f"✓ Matched {len(matched_data)} patients")

# Create aligned datasets
radiomics_features = np.array([d['radiomics_vector'] for d in matched_data])

survival_df = pd.DataFrame({
    'time': [d['time_years'] for d in matched_data],
    'event': [d['event'] for d in matched_data]
})

print(f"\n✅ DATA SUMMARY:")
print(f"   Patients: {len(matched_data)}")
print(f"   Events: {int(survival_df['event'].sum())}")
print(f"   Event rate: {survival_df['event'].mean()*100:.1f}%")
print(f"   Median survival: {survival_df['time'].median():.2f} years")


# ============================================================================
# STEP 2: GENERATE DYNAMIC FEATURES VIA MONTE CARLO
# ============================================================================

print("\n" + "="*80)
print("STEP 2: MONTE CARLO - GENERATE DYNAMIC FEATURES")
print("="*80)

# Standardize radiomics for Monte Carlo
scaler_mc = StandardScaler()
radiomics_standardized = scaler_mc.fit_transform(radiomics_features)

# Generate dynamic features
dynamic_df = generate_dynamic_features(
    radiomics_standardized,
    n_runs=N_MC_RUNS,
    T=T_MAX
)

# Standardize dynamic features
scaler_dynamic = StandardScaler()
dynamic_scaled = scaler_dynamic.fit_transform(dynamic_df)
dynamic_scaled_df = pd.DataFrame(dynamic_scaled, columns=dynamic_df.columns)


# ============================================================================
# STEP 3: FEATURE SELECTION FOR RADIOMICS
# ============================================================================

print("\n" + "="*80)
print("STEP 3: RADIOMICS FEATURE SELECTION")
print("="*80)

# Standardize radiomics
scaler_radiomics = StandardScaler()
radiomics_scaled = scaler_radiomics.fit_transform(radiomics_features)
radiomics_df = pd.DataFrame(radiomics_scaled, columns=feature_names)

# Select top features by correlation with event
print(f"\n🔍 Selecting top radiomics features by correlation with event...")
correlations = []
for col in radiomics_df.columns:
    corr = np.abs(np.corrcoef(radiomics_df[col], survival_df['event'])[0, 1])
    correlations.append((col, corr))

correlations.sort(key=lambda x: x[1], reverse=True)
top_n = min(10, len(correlations))  # Top 10 or all if less
top_radiomics_features = [c[0] for c in correlations[:top_n]]

print(f"   Selected top {top_n} radiomics features:")
for i, (feat, corr) in enumerate(correlations[:top_n], 1):
    print(f"   {i:2d}. {feat[:40]:40s} | corr: {corr:.4f}")

# ============================================================================
# STEP 4: TRAIN COX MODELS
# ============================================================================

print("\n" + "="*80)
print("STEP 4: TRAINING COX MODELS")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────
# MODEL 1: RADIOMICS ONLY
# ─────────────────────────────────────────────────────────────────────────

print("\n📊 MODEL 1: RADIOMICS ONLY")
print("-" * 80)

cox_radiomics = pd.concat([
    radiomics_df[top_radiomics_features].reset_index(drop=True),
    survival_df[['time', 'event']].reset_index(drop=True)
], axis=1)

cox_radiomics = cox_radiomics.dropna()

print(f"Features: {top_n} (top radiomics)")
print(f"Samples: {cox_radiomics.shape[0]}")

cph_radiomics = CoxPHFitter(penalizer=0.1)
c_index_radiomics = None

try:
    cph_radiomics.fit(cox_radiomics, duration_col='time', event_col='event', show_progress=False)
    c_index_radiomics = cph_radiomics.concordance_index_
    print(f"\n✓ Model trained successfully!")
    print(f"  C-Index: {c_index_radiomics:.4f}")
    
except Exception as e:
    print(f"✗ Error training model: {e}")

# ─────────────────────────────────────────────────────────────────────────
# MODEL 2: DYNAMIC FEATURES ONLY
# ─────────────────────────────────────────────────────────────────────────

print("\n📊 MODEL 2: DYNAMIC FEATURES (MONTE CARLO)")
print("-" * 80)

# Prepare Cox data
cox_dynamic = pd.concat([
    dynamic_scaled_df.reset_index(drop=True),
    survival_df[['time', 'event']].reset_index(drop=True)
], axis=1)

# Remove any NaN rows
cox_dynamic = cox_dynamic.dropna()

print(f"Features: 5 (dynamic from Monte Carlo)")
print(f"Samples: {cox_dynamic.shape[0]}")

# Train Cox model
cph_dynamic = CoxPHFitter(penalizer=0.1)
c_index_dynamic = None

try:
    cph_dynamic.fit(cox_dynamic, duration_col='time', event_col='event', show_progress=False)
    c_index_dynamic = cph_dynamic.concordance_index_
    
    print(f"\n✓ Model trained successfully!")
    print(f"  C-Index: {c_index_dynamic:.4f}")
    
except Exception as e:
    print(f"✗ Error training model: {e}")


# ============================================================================
# FINAL RESULTS - C-INDEX COMPARISON
# ============================================================================

print("\n" + "="*80)
print("🎯 RESULTS - C-INDEX COMPARISON")
print("="*80)

if c_index_radiomics is not None or c_index_dynamic is not None:
    
    # Create comparison visualization
    print(f"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                   COX SURVIVAL MODELS - C-INDEX COMPARISON                    ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  Model 1: RADIOMICS ONLY                                                     ║
║  ────────────────────────────────────────────────────────────────────────    ║
║    Features: Top {top_n} radiomics features                                              ║
║    C-Index: {c_index_radiomics:.4f}                                                        ║
║                                                                               ║
║                                                                               ║
║  Model 2: MONTE CARLO DYNAMIC FEATURES                                      ║
║  ────────────────────────────────────────────────────────────────────────    ║
║    Features: 5 (from Monte Carlo)                                            ║
║      • dynamic_auc                                                           ║
║      • dynamic_slope                                                         ║
║      • dynamic_time_half                                                     ║
║      • dynamic_peak                                                          ║
║      • dynamic_uncertainty                                                   ║
║    C-Index: {c_index_dynamic:.4f}                                                        ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  📊 COMPARISON:                                                               ║
║  ────────────────────────────────────────────────────────────────────────    ║""")
    
    if c_index_radiomics is not None and c_index_dynamic is not None:
        diff = c_index_dynamic - c_index_radiomics
        pct_change = (diff / c_index_radiomics) * 100 if c_index_radiomics > 0 else 0
        
        if diff > 0:
            symbol = "📈"
            direction = "BETTER"
        elif diff < 0:
            symbol = "📉"
            direction = "WORSE"
        else:
            symbol = "➡️"
            direction = "EQUAL"
        
        print(f"""║                                                                               ║
║    {symbol} Dynamic vs Radiomics: {diff:+.4f} ({pct_change:+.2f}%) {direction:10s}                                ║
║                                                                               ║""")
    
    print(f"""║  Dataset:                                                                     ║
║    • Total patients: {len(survival_df)}                                                        ║
║    • Events: {int(survival_df['event'].sum())}                                                          ║
║    • Event rate: {survival_df['event'].mean()*100:.1f}%                                              ║
║    • Median survival: {survival_df['time'].median():.2f} years                                              ║
║                                                                               ║
║  Monte Carlo Configuration:                                                  ║
║    • Runs per patient: {N_MC_RUNS}                                                    ║
║    • Time horizon: {T_MAX} years                                                     ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

    # Save results
    result_dict = {
        "model_1_radiomics_only": {
            "name": "Cox Proportional Hazards (Radiomics Only)",
            "n_features": top_n,
            "features": top_radiomics_features,
            "c_index": float(c_index_radiomics) if c_index_radiomics else None,
            "n_patients": int(len(cox_radiomics)) if c_index_radiomics else None,
            "n_events": int(cox_radiomics['event'].sum()) if c_index_radiomics else None
        },
        "model_2_dynamic_only": {
            "name": "Cox Proportional Hazards (Dynamic Features)",
            "n_features": 5,
            "features": [
                "dynamic_auc",
                "dynamic_slope", 
                "dynamic_time_half",
                "dynamic_peak",
                "dynamic_uncertainty"
            ],
            "c_index": float(c_index_dynamic) if c_index_dynamic else None,
            "n_patients": int(len(cox_dynamic)) if c_index_dynamic else None,
            "n_events": int(cox_dynamic['event'].sum()) if c_index_dynamic else None
        },
        "dataset_info": {
            "total_patients": len(survival_df),
            "total_events": int(survival_df['event'].sum()),
            "event_rate": float(survival_df['event'].mean()),
            "median_survival_years": float(survival_df['time'].median()),
            "mean_survival_years": float(survival_df['time'].mean())
        },
        "monte_carlo": {
            "runs_per_patient": N_MC_RUNS,
            "time_horizon_years": T_MAX
        }
    }
    
    # Add comparison if both models trained
    if c_index_radiomics is not None and c_index_dynamic is not None:
        diff = c_index_dynamic - c_index_radiomics
        pct_change = (diff / c_index_radiomics) * 100 if c_index_radiomics > 0 else 0
        result_dict["comparison"] = {
            "difference": float(diff),
            "percent_change": float(pct_change),
            "better_model": "dynamic" if diff > 0 else ("radiomics" if diff < 0 else "equal")
        }
    
    result_path = os.path.join(OUTPUT_DIR, 'c_index_comparison.json')
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=2)
    
    print(f"\n💾 Results saved: {result_path}")

else:
    print("✗ Model training failed!")

print("\n" + "="*80 + "\n")


╔═══════════════════════════════════════════════════════════════════════════════╗
║      SURVIVAL ANALYSIS - MONTE CARLO (C-INDEX ONLY)                          ║
║                   Patient ID Matching + Years (Real Time)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝


CONFIGURATION:
  • Monte Carlo runs: 500
  • Time horizon: 15 years
  • Random seed: 42

STEP 1: LOADING DATA

📂 Radiomics: visualization_output/features_normalized_standard.npy
   Shape: (421, 261)
   Patient IDs: 421
📂 Clinical: nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv

STEP 2: PATIENT ID MATCHING

🔍 Matching patients...
✓ Matched 421 patients

✅ DATA SUMMARY:
   Patients: 421
   Events: 373
   Event rate: 88.6%
   Median survival: 1.50 years

STEP 2: MONTE CARLO - GENERATE DYNAMIC FEATURES

⚙️  Generating dynamic features for 421 patients...
  Processing: 100/421
  Processing: 200/421
  Processing: 300/421
  Processing: 400/421
  ✓ Generated

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Survival Analysis with Monte Carlo - 3 MODELS COMPARISON
=========================================================

So sánh 3 mô hình Cox:
1. Radiomics only
2. Dynamic Features only (Monte Carlo)
3. Combined (Radiomics + Dynamic) ✨ KẾT HỢP

Author: Claude
Date: 2025-11-18
"""

import os
import json
import numpy as np
import pandas as pd
from typing import Dict
import warnings

from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

print("""
╔═══════════════════════════════════════════════════════════════════════════════╗
║    SURVIVAL ANALYSIS - 3 MODELS COMPARISON (INCLUDING COMBINED)              ║
║                   Patient ID Matching + Years (Real Time)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

# ============================================================================
# CONFIGURATION
# ============================================================================

RANDOM_SEED = 42
N_MC_RUNS = 500  # Monte Carlo runs per patient
T_MAX = 15       # Time horizon in YEARS

OUTPUT_DIR = 'visualization_output/survival_analysis_3models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nCONFIGURATION:")
print(f"  • Monte Carlo runs: {N_MC_RUNS}")
print(f"  • Time horizon: {T_MAX} years")
print(f"  • Random seed: {RANDOM_SEED}")


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def days_to_years(days: float) -> float:
    """Convert days to years."""
    if pd.isna(days):
        return np.nan
    return days / 365.25


def sigmoid(t: np.ndarray, L: float, k: float, t0: float) -> np.ndarray:
    """Sigmoid function for response modeling."""
    return L / (1 + np.exp(-k * (t - t0)))


def monte_carlo_from_radiomics(radiomics_vector: np.ndarray,
                               n_runs: int = 500,
                               T: int = 15,
                               seed: int = None) -> Dict:
    """
    Transform static radiomics into dynamic features via Monte Carlo.
    Returns: auc, slope, time_half, peak, uncertainty
    """
    if seed is not None:
        np.random.seed(seed)

    x = radiomics_vector
    t = np.linspace(0, T, T)
    curves = []

    for _ in range(n_runs):
        # Random projection weights
        w_L = np.random.randn(len(x))
        w_k = np.random.randn(len(x))
        w_t0 = np.random.randn(len(x))

        # Generate sigmoid parameters
        L = np.dot(w_L, x)
        L = abs(L) + 1e-3

        k_raw = np.dot(w_k, x)
        k = 0.1 + 0.9 * (1 / (1 + np.exp(-k_raw)))

        t0_raw = np.dot(w_t0, x)
        t0 = 0.5 + 13.5 * (1 / (1 + np.exp(-t0_raw)))

        # Generate sigmoid curve
        S = sigmoid(t, L, k, t0)
        curves.append(S)

    curves = np.array(curves)

    # Aggregate statistics
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)

    # Extract dynamic features
    gradient = np.gradient(mean_curve)
    slope = np.max(gradient)
    auc = np.trapz(mean_curve, t)

    half_max = 0.5 * mean_curve.max()
    time_half_idx = np.argmax(mean_curve >= half_max)
    time_half = t[time_half_idx] if time_half_idx > 0 else 0

    peak_idx = np.argmax(gradient)
    peak = mean_curve[peak_idx]

    uncertainty = std_curve.mean()

    return {
        "auc": auc,
        "slope": slope,
        "time_half": time_half,
        "peak": peak,
        "uncertainty": uncertainty
    }


def generate_dynamic_features(radiomics_features: np.ndarray,
                             n_runs: int = 500,
                             T: int = 15) -> pd.DataFrame:
    """Generate dynamic features for all patients via Monte Carlo."""
    n_patients = radiomics_features.shape[0]
    dynamic_features = []

    print(f"\n⚙️  Generating dynamic features for {n_patients} patients...")

    for i in range(n_patients):
        if (i + 1) % 100 == 0:
            print(f"  Processing: {i+1}/{n_patients}")

        radiomics_vector = radiomics_features[i]
        result = monte_carlo_from_radiomics(
            radiomics_vector,
            n_runs=n_runs,
            T=T,
            seed=RANDOM_SEED + i
        )

        dynamic_features.append([
            result["auc"],
            result["slope"],
            result["time_half"],
            result["peak"],
            result["uncertainty"]
        ])

    df = pd.DataFrame(
        dynamic_features,
        columns=["dynamic_auc", "dynamic_slope", "dynamic_time_half",
                 "dynamic_peak", "dynamic_uncertainty"]
    )

    print(f"  ✓ Generated: {df.shape}")
    return df


# ============================================================================
# STEP 1: LOAD DATA WITH PATIENT ID MATCHING
# ============================================================================

print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# Load radiomics features
features_path = 'visualization_output/features_normalized_standard.npy'
if not os.path.exists(features_path):
    features_path = 'visualization_output/radiomic_features_440.npy'

print(f"\n📂 Radiomics: {features_path}")
radiomics_features_all = np.load(features_path)
print(f"   Shape: {radiomics_features_all.shape}")

# Load feature names
names_path = 'visualization_output/feature_names_normalized_standard.json'
with open(names_path, 'r') as f:
    feature_names = json.load(f)

# Load patient IDs
ids_path = 'visualization_output/patient_ids.json'
with open(ids_path, 'r') as f:
    patient_ids_all = json.load(f)
print(f"   Patient IDs: {len(patient_ids_all)}")

# Load clinical data
clinical_paths = [
    "nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv",
    "NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv",
]

clinical_data = None
for path in clinical_paths:
    if os.path.exists(path):
        clinical_data = pd.read_csv(path)
        print(f"📂 Clinical: {path}")
        break

if clinical_data is None:
    raise FileNotFoundError("Clinical data not found!")

# ============================================================================
# PATIENT ID MATCHING
# ============================================================================

print("\n" + "="*80)
print("STEP 2: PATIENT ID MATCHING")
print("="*80)

# Find patient ID column
patient_id_col = None
for col in clinical_data.columns:
    if 'patient' in col.lower() and 'id' in col.lower():
        patient_id_col = col
        break

if patient_id_col is None:
    clinical_data['PatientID'] = clinical_data.index.astype(str)
    patient_id_col = 'PatientID'

clinical_data['PatientID_str'] = clinical_data[patient_id_col].astype(str)
patient_ids_radiomics = [str(pid) for pid in patient_ids_all]

# Find survival columns
time_col = [c for c in clinical_data.columns if 'survival.time' in c.lower()][0]
event_col = [c for c in clinical_data.columns if 'event' in c.lower() or 'deadstatus' in c.lower()][0]

# Match patients
print("\n🔍 Matching patients...")
matched_data = []

for i, pid in enumerate(patient_ids_radiomics):
    clinical_row = clinical_data[clinical_data['PatientID_str'] == pid]

    if len(clinical_row) == 0:
        if i < len(clinical_data):
            clinical_row = clinical_data.iloc[[i]]
        else:
            continue

    if len(clinical_row) == 0:
        continue

    time_days = clinical_row[time_col].values[0]
    event = clinical_row[event_col].values[0]

    if pd.isna(time_days) or pd.isna(event):
        continue

    time_years = days_to_years(time_days)
    radiomics_vector = radiomics_features_all[i]

    matched_data.append({
        'patient_id': pid,
        'time_years': time_years,
        'event': event,
        'radiomics_vector': radiomics_vector
    })

print(f"✓ Matched {len(matched_data)} patients")

# Create aligned datasets
radiomics_features = np.array([d['radiomics_vector'] for d in matched_data])

survival_df = pd.DataFrame({
    'time': [d['time_years'] for d in matched_data],
    'event': [d['event'] for d in matched_data]
})

print(f"\n✅ DATA SUMMARY:")
print(f"   Patients: {len(matched_data)}")
print(f"   Events: {int(survival_df['event'].sum())}")
print(f"   Event rate: {survival_df['event'].mean()*100:.1f}%")
print(f"   Median survival: {survival_df['time'].median():.2f} years")


# ============================================================================
# STEP 2: GENERATE DYNAMIC FEATURES VIA MONTE CARLO
# ============================================================================

print("\n" + "="*80)
print("STEP 2: MONTE CARLO - GENERATE DYNAMIC FEATURES")
print("="*80)

# Standardize radiomics for Monte Carlo
scaler_mc = StandardScaler()
radiomics_standardized = scaler_mc.fit_transform(radiomics_features)

# Generate dynamic features
dynamic_df = generate_dynamic_features(
    radiomics_standardized,
    n_runs=N_MC_RUNS,
    T=T_MAX
)

# Standardize dynamic features
scaler_dynamic = StandardScaler()
dynamic_scaled = scaler_dynamic.fit_transform(dynamic_df)
dynamic_scaled_df = pd.DataFrame(dynamic_scaled, columns=dynamic_df.columns)


# ============================================================================
# STEP 3: FEATURE SELECTION FOR RADIOMICS
# ============================================================================

print("\n" + "="*80)
print("STEP 3: RADIOMICS FEATURE SELECTION")
print("="*80)

# Standardize radiomics
scaler_radiomics = StandardScaler()
radiomics_scaled = scaler_radiomics.fit_transform(radiomics_features)
radiomics_df = pd.DataFrame(radiomics_scaled, columns=feature_names)

# Select top features by correlation with event
print(f"\n🔍 Selecting top radiomics features by correlation with event...")
correlations = []
for col in radiomics_df.columns:
    corr = np.abs(np.corrcoef(radiomics_df[col], survival_df['event'])[0, 1])
    correlations.append((col, corr))

correlations.sort(key=lambda x: x[1], reverse=True)
top_n = max(10, len(correlations))  # Top 10 or all if less
top_radiomics_features = [c[0] for c in correlations[:top_n]]

print(f"   Selected top {top_n} radiomics features:")
for i, (feat, corr) in enumerate(correlations[:top_n], 1):
    print(f"   {i:2d}. {feat[:40]:40s} | corr: {corr:.4f}")

# ============================================================================
# STEP 4: TRAIN 3 COX MODELS
# ============================================================================

print("\n" + "="*80)
print("STEP 4: TRAINING 3 COX MODELS")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────
# MODEL 1: RADIOMICS ONLY
# ─────────────────────────────────────────────────────────────────────────

print("\n📊 MODEL 1: RADIOMICS ONLY")
print("-" * 80)

cox_radiomics = pd.concat([
    radiomics_df[top_radiomics_features].reset_index(drop=True),
    survival_df[['time', 'event']].reset_index(drop=True)
], axis=1)

cox_radiomics = cox_radiomics.dropna()

print(f"Features: {top_n} (top radiomics)")
print(f"Samples: {cox_radiomics.shape[0]}")

cph_radiomics = CoxPHFitter(penalizer=0.1)
c_index_radiomics = None

try:
    cph_radiomics.fit(cox_radiomics, duration_col='time', event_col='event', show_progress=False)
    c_index_radiomics = cph_radiomics.concordance_index_
    print(f"\n✓ Model trained successfully!")
    print(f"  C-Index: {c_index_radiomics:.4f}")
    
except Exception as e:
    print(f"✗ Error training model: {e}")

# ─────────────────────────────────────────────────────────────────────────
# MODEL 2: DYNAMIC FEATURES ONLY
# ─────────────────────────────────────────────────────────────────────────

print("\n📊 MODEL 2: DYNAMIC FEATURES (MONTE CARLO)")
print("-" * 80)

cox_dynamic = pd.concat([
    dynamic_scaled_df.reset_index(drop=True),
    survival_df[['time', 'event']].reset_index(drop=True)
], axis=1)

cox_dynamic = cox_dynamic.dropna()

print(f"Features: 5 (dynamic from Monte Carlo)")
print(f"Samples: {cox_dynamic.shape[0]}")

cph_dynamic = CoxPHFitter(penalizer=0.1)
c_index_dynamic = None

try:
    cph_dynamic.fit(cox_dynamic, duration_col='time', event_col='event', show_progress=False)
    c_index_dynamic = cph_dynamic.concordance_index_
    
    print(f"\n✓ Model trained successfully!")
    print(f"  C-Index: {c_index_dynamic:.4f}")
    
except Exception as e:
    print(f"✗ Error training model: {e}")

# ─────────────────────────────────────────────────────────────────────────
# MODEL 3: COMBINED (RADIOMICS + DYNAMIC FEATURES)
# ─────────────────────────────────────────────────────────────────────────

print("\n📊 MODEL 3: COMBINED (RADIOMICS + DYNAMIC) ✨")
print("-" * 80)

cox_combined = pd.concat([
    radiomics_df[top_radiomics_features].reset_index(drop=True),
    dynamic_scaled_df.reset_index(drop=True),
    survival_df[['time', 'event']].reset_index(drop=True)
], axis=1)

cox_combined = cox_combined.dropna()

print(f"Features: {top_n} radiomics + 5 dynamic = {top_n + 5} total")
print(f"Samples: {cox_combined.shape[0]}")

cph_combined = CoxPHFitter(penalizer=0.1)
c_index_combined = None

try:
    cph_combined.fit(cox_combined, duration_col='time', event_col='event', show_progress=False)
    c_index_combined = cph_combined.concordance_index_
    
    print(f"\n✓ Model trained successfully!")
    print(f"  C-Index: {c_index_combined:.4f}")
    
except Exception as e:
    print(f"✗ Error training model: {e}")


# ============================================================================
# FINAL RESULTS - 3 MODEL COMPARISON
# ============================================================================

print("\n" + "="*80)
print("🎯 RESULTS - 3 MODEL COMPARISON")
print("="*80)

if c_index_radiomics is not None or c_index_dynamic is not None or c_index_combined is not None:
    
    print(f"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║           COX SURVIVAL MODELS - 3 MODEL C-INDEX COMPARISON                    ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  Model 1: RADIOMICS ONLY                                                     ║
║  ────────────────────────────────────────────────────────────────────────    ║
║    Features: Top {top_n} radiomics features                                              ║
║    C-Index: {c_index_radiomics:.4f}                                                        ║
║                                                                               ║
║                                                                               ║
║  Model 2: MONTE CARLO DYNAMIC FEATURES                                      ║
║  ────────────────────────────────────────────────────────────────────────    ║
║    Features: 5 (from Monte Carlo)                                            ║
║      • dynamic_auc                                                           ║
║      • dynamic_slope                                                         ║
║      • dynamic_time_half                                                     ║
║      • dynamic_peak                                                          ║
║      • dynamic_uncertainty                                                   ║
║    C-Index: {c_index_dynamic:.4f}                                                        ║
║                                                                               ║
║                                                                               ║
║  Model 3: COMBINED (RADIOMICS + DYNAMIC) ✨                                 ║
║  ────────────────────────────────────────────────────────────────────────    ║
║    Features: {top_n + 5} total ({top_n} radiomics + 5 dynamic)                                       ║
║    C-Index: {c_index_combined:.4f}                                                        ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  📊 COMPARISON METRICS:                                                       ║
║  ────────────────────────────────────────────────────────────────────────    ║""")
    
    if c_index_radiomics is not None and c_index_dynamic is not None:
        diff_dyn = c_index_dynamic - c_index_radiomics
        pct_dyn = (diff_dyn / c_index_radiomics) * 100 if c_index_radiomics > 0 else 0
        symbol_dyn = "📈" if diff_dyn > 0 else ("📉" if diff_dyn < 0 else "➡️")
        direction_dyn = "BETTER" if diff_dyn > 0 else ("WORSE" if diff_dyn < 0 else "EQUAL")
        print(f"""║                                                                               ║
║    Dynamic vs Radiomics:                                                      ║
║    {symbol_dyn} {diff_dyn:+.4f} ({pct_dyn:+.2f}%) {direction_dyn:10s}                                       ║""")
    
    if c_index_radiomics is not None and c_index_combined is not None:
        diff_comb_rad = c_index_combined - c_index_radiomics
        pct_comb_rad = (diff_comb_rad / c_index_radiomics) * 100 if c_index_radiomics > 0 else 0
        symbol_comb_rad = "📈" if diff_comb_rad > 0 else ("📉" if diff_comb_rad < 0 else "➡️")
        direction_comb_rad = "BETTER" if diff_comb_rad > 0 else ("WORSE" if diff_comb_rad < 0 else "EQUAL")
        print(f"""║                                                                               ║
║    Combined vs Radiomics:                                                     ║
║    {symbol_comb_rad} {diff_comb_rad:+.4f} ({pct_comb_rad:+.2f}%) {direction_comb_rad:10s}                                      ║""")
    
    if c_index_dynamic is not None and c_index_combined is not None:
        diff_comb_dyn = c_index_combined - c_index_dynamic
        pct_comb_dyn = (diff_comb_dyn / c_index_dynamic) * 100 if c_index_dynamic > 0 else 0
        symbol_comb_dyn = "📈" if diff_comb_dyn > 0 else ("📉" if diff_comb_dyn < 0 else "➡️")
        direction_comb_dyn = "BETTER" if diff_comb_dyn > 0 else ("WORSE" if diff_comb_dyn < 0 else "EQUAL")
        print(f"""║                                                                               ║
║    Combined vs Dynamic:                                                       ║
║    {symbol_comb_dyn} {diff_comb_dyn:+.4f} ({pct_comb_dyn:+.2f}%) {direction_comb_dyn:10s}                                        ║""")
    
    print(f"""║                                                                               ║
║  Dataset:                                                                     ║
║    • Total patients: {len(survival_df)}                                                        ║
║    • Events: {int(survival_df['event'].sum())}                                                          ║
║    • Event rate: {survival_df['event'].mean()*100:.1f}%                                              ║
║    • Median survival: {survival_df['time'].median():.2f} years                                              ║
║                                                                               ║
║  Monte Carlo Configuration:                                                  ║
║    • Runs per patient: {N_MC_RUNS}                                                    ║
║    • Time horizon: {T_MAX} years                                                     ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

    # Save results
    result_dict = {
        "model_1_radiomics_only": {
            "name": "Cox Proportional Hazards (Radiomics Only)",
            "n_features": top_n,
            "features": top_radiomics_features,
            "c_index": float(c_index_radiomics) if c_index_radiomics else None,
            "n_patients": int(len(cox_radiomics)) if c_index_radiomics else None,
            "n_events": int(cox_radiomics['event'].sum()) if c_index_radiomics else None
        },
        "model_2_dynamic_only": {
            "name": "Cox Proportional Hazards (Dynamic Features)",
            "n_features": 5,
            "features": [
                "dynamic_auc",
                "dynamic_slope", 
                "dynamic_time_half",
                "dynamic_peak",
                "dynamic_uncertainty"
            ],
            "c_index": float(c_index_dynamic) if c_index_dynamic else None,
            "n_patients": int(len(cox_dynamic)) if c_index_dynamic else None,
            "n_events": int(cox_dynamic['event'].sum()) if c_index_dynamic else None
        },
        "model_3_combined": {
            "name": "Cox Proportional Hazards (Radiomics + Dynamic Combined)",
            "n_features": top_n + 5,
            "features": top_radiomics_features + [
                "dynamic_auc",
                "dynamic_slope", 
                "dynamic_time_half",
                "dynamic_peak",
                "dynamic_uncertainty"
            ],
            "c_index": float(c_index_combined) if c_index_combined else None,
            "n_patients": int(len(cox_combined)) if c_index_combined else None,
            "n_events": int(cox_combined['event'].sum()) if c_index_combined else None
        },
        "dataset_info": {
            "total_patients": len(survival_df),
            "total_events": int(survival_df['event'].sum()),
            "event_rate": float(survival_df['event'].mean()),
            "median_survival_years": float(survival_df['time'].median()),
            "mean_survival_years": float(survival_df['time'].mean())
        },
        "monte_carlo": {
            "runs_per_patient": N_MC_RUNS,
            "time_horizon_years": T_MAX
        }
    }
    
    # Add comparisons
    if c_index_radiomics is not None and c_index_dynamic is not None:
        diff = c_index_dynamic - c_index_radiomics
        pct_change = (diff / c_index_radiomics) * 100 if c_index_radiomics > 0 else 0
        result_dict["comparison_dynamic_vs_radiomics"] = {
            "difference": float(diff),
            "percent_change": float(pct_change),
            "better_model": "dynamic" if diff > 0 else ("radiomics" if diff < 0 else "equal")
        }
    
    if c_index_radiomics is not None and c_index_combined is not None:
        diff = c_index_combined - c_index_radiomics
        pct_change = (diff / c_index_radiomics) * 100 if c_index_radiomics > 0 else 0
        result_dict["comparison_combined_vs_radiomics"] = {
            "difference": float(diff),
            "percent_change": float(pct_change),
            "better_model": "combined" if diff > 0 else ("radiomics" if diff < 0 else "equal")
        }
    
    if c_index_dynamic is not None and c_index_combined is not None:
        diff = c_index_combined - c_index_dynamic
        pct_change = (diff / c_index_dynamic) * 100 if c_index_dynamic > 0 else 0
        result_dict["comparison_combined_vs_dynamic"] = {
            "difference": float(diff),
            "percent_change": float(pct_change),
            "better_model": "combined" if diff > 0 else ("dynamic" if diff < 0 else "equal")
        }
    
    result_path = os.path.join(OUTPUT_DIR, '3models_comparison.json')
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=2)
    
    print(f"\n💾 Results saved: {result_path}")

else:
    print("✗ Model training failed!")

print("\n" + "="*80 + "\n")


╔═══════════════════════════════════════════════════════════════════════════════╗
║    SURVIVAL ANALYSIS - 3 MODELS COMPARISON (INCLUDING COMBINED)              ║
║                   Patient ID Matching + Years (Real Time)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝


CONFIGURATION:
  • Monte Carlo runs: 500
  • Time horizon: 15 years
  • Random seed: 42

STEP 1: LOADING DATA

📂 Radiomics: visualization_output/features_normalized_standard.npy
   Shape: (421, 261)
   Patient IDs: 421
📂 Clinical: nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv

STEP 2: PATIENT ID MATCHING

🔍 Matching patients...
✓ Matched 421 patients

✅ DATA SUMMARY:
   Patients: 421
   Events: 373
   Event rate: 88.6%
   Median survival: 1.50 years

STEP 2: MONTE CARLO - GENERATE DYNAMIC FEATURES

⚙️  Generating dynamic features for 421 patients...
  Processing: 100/421
  Processing: 200/421
  Processing: 300/421
  Processing: 400/421
  ✓ Generated


✓ Model trained successfully!
  C-Index: 0.6693

📊 MODEL 2: DYNAMIC FEATURES (MONTE CARLO)
--------------------------------------------------------------------------------
Features: 5 (dynamic from Monte Carlo)
Samples: 421

✓ Model trained successfully!
  C-Index: 0.5177

📊 MODEL 3: COMBINED (RADIOMICS + DYNAMIC) ✨
--------------------------------------------------------------------------------
Features: 261 radiomics + 5 dynamic = 266 total
Samples: 421

✓ Model trained successfully!
  C-Index: 0.6692

🎯 RESULTS - 3 MODEL COMPARISON

╔═══════════════════════════════════════════════════════════════════════════════╗
║           COX SURVIVAL MODELS - 3 MODEL C-INDEX COMPARISON                    ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  Model 1: RADIOMICS ONLY                                                     ║
║  ─────────────────────────────────────────────